<a href="https://colab.research.google.com/github/Deangr-econ/Stock_volatility/blob/main/Stock_Vol.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount('/content/drive')



cov = pd.read_csv("/content/drive/Othercomputers/My Laptop/Documents/Maastricht University/Master/Thesis/Volare Data/realized_covariance_stocks_20261030/realized_covariance_stocks_20261030.csv")
cov['date'] = pd.to_datetime(cov['date'])
cov.info()
print(f"Nan-Values:",cov.isna().sum().sum())

# Filter out duplicate pairs since Cov(A, B) = Cov(B, A)
# We'll keep only pairs where asset1 <= asset2 to avoid redundant columns
cov_filtered = cov[cov['asset1'] <= cov['asset2']].copy()

# Create a unique feature column name for each covariance combination
cov_filtered['pair'] = cov_filtered['asset1'] + "_" + cov_filtered['asset2']

# Pivot the dataset: rows = date, columns = pair, values = rcov
X_df = cov_filtered.pivot(index='date', columns='pair', values='rcov')

# Check for any remaining missing values (if any asset pair was missing on a specific date)
missing_count = X_df.isna().sum().sum()
if missing_count > 0:
    # Forward fill then backward fill to handle any isolated missing values safely
    X_df = X_df.ffill().bfill()

print(f"X Matrix shape: {X_df.shape} (Rows/Dates: {X_df.shape[0]}, Columns/Pairs: {X_df.shape[1]})")
print(f"Number of imputed missing values: {missing_count}")

# Display the first few rows of the transformed design matrix
X_df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2439500 entries, 0 to 2439499
Data columns (total 9 columns):
 #   Column    Dtype         
---  ------    -----         
 0   date      datetime64[ns]
 1   asset1    object        
 2   asset2    object        
 3   rcov      float64       
 4   rbpcov    float64       
 5   rscov_p   float64       
 6   rscov_n   float64       
 7   rscov_mp  float64       
 8   rscov_mn  float64       
dtypes: datetime64[ns](1), float64(6), object(2)
memory usage: 167.5+ MB
Nan-Values: 0
X Matrix shape: (2975, 820) (Rows/Dates: 2975, Columns/Pairs: 820)
Number of imputed missing values: 0


pair,AAPL_AAPL,AAPL_ADBE,AAPL_AMD,AAPL_AMGN,AAPL_AMZN,AAPL_AXP,AAPL_BA,AAPL_CAT,AAPL_CRM,AAPL_CSCO,...,VZ_VZ,VZ_WMT,VZ_XOM,V_V,V_VZ,V_WMT,V_XOM,WMT_WMT,WMT_XOM,XOM_XOM
date,,,,,,,,,,,,,,,,,,,,,
2015-01-02,0.000275,0.000086,0.000075,0.000120,0.000092,0.000086,0.000051,0.000056,0.000125,0.000096,...,0.000093,0.000032,0.000034,0.000226,0.000039,0.000039,0.000020,0.000080,0.000021,0.000160
2015-01-05,0.000278,0.000111,0.000042,0.000143,0.000084,0.000093,0.000078,0.000035,0.000165,0.000134,...,0.000101,0.000030,0.000029,0.000145,0.000023,0.000047,0.000041,0.000095,0.000031,0.000180
2015-01-06,0.000370,0.000144,0.000102,0.000208,0.000166,0.000141,0.000100,0.000115,0.000200,0.000143,...,0.000166,0.000050,0.000028,0.000239,0.000047,0.000058,0.000088,0.000120,0.000047,0.000228
2015-01-07,0.000210,0.000117,0.000046,0.000154,0.000129,0.000105,0.000067,0.000090,0.000166,0.000120,...,0.000115,0.000040,0.000048,0.000138,0.000051,0.000060,0.000058,0.000134,0.000044,0.000191
2015-01-08,0.000146,0.000047,-0.000046,0.000007,0.000018,0.000026,0.000022,0.000018,0.000050,0.000021,...,0.000101,0.000016,0.000027,0.000098,0.000024,0.000014,0.000024,0.000112,0.000013,0.000141


In [51]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import time
from numba import njit, prange

class FactorModelEvaluator:
    """
    An OOP-style Evaluator to run out-of-sample factor forecasting evaluations
    using either expanding or rolling window approaches, decoupling calculation of error matrices
    from their subsequent metrics evaluation and plotting.
    """
    def __init__(self, df, r_max=820, n_eval=800, window_size=1000):
        self.df = df
        self.r_max = r_max
        self.n_eval = n_eval
        self.window_size = window_size
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.eval_dates = df.index[-n_eval:]

    def compute_expanding_errors(self):
        """
        Runs expanding window SVD and regression, returning a dictionary of
        error DataFrames (Actual - Forecast) indexed by evaluation dates for each r.
        Fully vectorized across factor dimensions r using incremental matrix calculations.
        """
        print(f"Computing expanding window errors on: {self.device}")
        X_raw = torch.tensor(self.df.values, dtype=torch.float32, device=self.device)
        T, N = X_raw.shape
        start_idx = T - self.n_eval

        errors_tensor = torch.empty((self.r_max, self.n_eval, N), dtype=torch.float32, device=self.device)

        for idx, t in enumerate(range(start_idx, T)):
            X_win = X_raw[:t]
            mean_t = X_win.mean(dim=0, keepdim=True)
            std_t = X_win.std(dim=0, keepdim=True)
            std_t = torch.clamp(std_t, min=1e-8)
            X_norm = (X_win - mean_t) / std_t

            # SVD on standard scale
            U, S, _ = torch.linalg.svd(X_norm, full_matrices=False)
            F_full = U[:, :self.r_max] * S[:self.r_max].unsqueeze(0)

            F_train = F_full[:-1, :]
            Y_train = X_norm[1:, :]
            f_latest = F_full[-1:, :]

            # Solve for all r at once using cumulative/triangular structure or batched solvers
            r_indices = torch.arange(1, self.r_max + 1, device=self.device)

            # Generate masking matrix to solve nested OLS systems in parallel
            FtF = F_train.T @ F_train
            FtY = F_train.T @ Y_train

            # Solve OLS independently per r using a batched solver
            LHS = torch.zeros((self.r_max, self.r_max, self.r_max), dtype=torch.float32, device=self.device)
            RHS = torch.zeros((self.r_max, self.r_max, N), dtype=torch.float32, device=self.device)
            F_lat_batched = torch.zeros((self.r_max, 1, self.r_max), dtype=torch.float32, device=self.device)

            for r_idx in range(self.r_max):
                r = r_idx + 1
                LHS[r_idx, :r, :r] = FtF[:r, :r] + 1e-6 * torch.eye(r, device=self.device)
                if r < self.r_max:
                    LHS[r_idx, r:, r:] = torch.eye(self.r_max - r, device=self.device)
                RHS[r_idx, :r, :] = FtY[:r, :]
                F_lat_batched[r_idx, 0, :r] = f_latest[0, :r]

            # Solve the entire batch
            Beta_all = torch.linalg.solve(LHS, RHS)

            # Predict and evaluate error in one batched multiplication
            x_pred_std_all = torch.bmm(F_lat_batched, Beta_all).squeeze(1) # Shape: (r_max, N)

            # Unstandardize the forecast to the raw scale
            x_pred_raw_all = x_pred_std_all * std_t + mean_t

            # Store forecast error directly based on unscaled actual values
            errors_tensor[:, idx] = X_raw[t : t + 1] - x_pred_raw_all

        errors_np = errors_tensor.cpu().numpy()
        errors_by_r = {
            r: pd.DataFrame(errors_np[r - 1], index=self.eval_dates, columns=self.df.columns)
            for r in range(1, self.r_max + 1)
        }
        return errors_by_r

    def compute_rolling_errors(self):
        """
        Runs rolling window SVD and regression, returning a dictionary of
        error DataFrames (Actual - Forecast) indexed by evaluation dates for each r.
        """
        print(f"Computing rolling window errors on: {self.device}")
        df_aligned = self.df.iloc[-(self.window_size + self.n_eval) :]

        raw_tensor = torch.tensor(df_aligned.values, dtype=torch.float32, device=self.device)

        X_raw_batched = raw_tensor.unfold(0, self.window_size, 1).permute(0, 2, 1)[:-1].contiguous()
        B, W, N = X_raw_batched.shape

        win_mean = X_raw_batched.mean(dim=1, keepdim=True)              # (B, 1, N)
        win_std = X_raw_batched.std(dim=1, unbiased=True, keepdim=True) # (B, 1, N)
        win_std = torch.clamp(win_std, min=1e-8)

        X_batched = (X_raw_batched - win_mean) / win_std

        X_actual_raw = raw_tensor[self.window_size:, :].unsqueeze(1)    # (B, 1, N)

        U_batched, S_batched, _ = torch.linalg.svd(X_batched, full_matrices=False)
        Y_train = X_batched[:, 1:, :]

        errors_tensor = torch.empty((self.r_max, B, N), dtype=torch.float32, device=self.device)

        for r in range(1, self.r_max + 1):
            F_batched = U_batched[:, :, :r] * S_batched[:, :r].unsqueeze(1)
            F_train = F_batched[:, :-1, :]
            F_latest = F_batched[:, -1:, :]

            Ft = F_train.transpose(1, 2)
            FtF = torch.bmm(Ft, F_train)
            FtY = torch.bmm(Ft, Y_train)

            FtF_reg = FtF + 1e-6 * torch.eye(r, device=self.device).unsqueeze(0)
            Beta = torch.bmm(torch.linalg.inv(FtF_reg), FtY)

            X_pred_std = torch.bmm(F_latest, Beta)

            # Unstandardize
            X_pred_raw = X_pred_std * win_std + win_mean

            errors_tensor[r - 1] = (X_actual_raw - X_pred_raw).squeeze(1)

        errors_np = errors_tensor.cpu().numpy()
        errors_by_r = {
            r: pd.DataFrame(errors_np[r - 1], index=self.eval_dates, columns=self.df.columns)
            for r in range(1, self.r_max + 1)
        }
        return errors_by_r

    @staticmethod
    def evaluate_performance(errors_by_r, start_idx=0, end_idx=None):
        metrics = []
        for r, err_df in errors_by_r.items():
            err_vals = err_df.iloc[start_idx:end_idx].values
            rmse = np.sqrt(np.mean(err_vals ** 2))
            mae = np.mean(np.abs(err_vals))
            metrics.append({'r': r, 'RMSE': rmse, 'MAE': mae})
        return pd.DataFrame(metrics).set_index('r')


@njit(fastmath=True)
def numba_single_lasso_with_warm_start(G, xty, alpha, beta_init, max_iter=1000, tol=1e-5):
    """
    Performs coordinate descent for a single target and alpha using a precomputed Gram matrix
    and a warm-start coefficient vector 'beta_init'.
    """
    N = G.shape[0]
    beta = beta_init.copy()

    r = xty.copy()
    for j in range(N):
        if beta[j] != 0.0:
            for k in range(N):
                r[k] -= beta[j] * G[k, j]

    for iteration in range(max_iter):
        max_change = 0.0
        for j in range(N):
            old_beta_j = beta[j]
            rho = r[j] + old_beta_j

            if rho > alpha:
                new_beta_j = rho - alpha
            elif rho < -alpha:
                new_beta_j = rho + alpha
            else:
                new_beta_j = 0.0

            diff = new_beta_j - old_beta_j
            if diff != 0.0:
                beta[j] = new_beta_j
                for k in range(N):
                    r[k] -= diff * G[k, j]

                change = abs(diff)
                if change > max_change:
                    max_change = change

        if max_change < tol:
            break

    return beta

@njit(parallel=True, fastmath=True)
def numba_parallel_lasso(X, Y, alphas, max_iter=1000, tol=1e-5):
    """
    Highly optimized parallel LASSO solver.
    """
    M, N = X.shape
    n_alphas = len(alphas)
    coefs = np.zeros((N, N, n_alphas), dtype=np.float64)

    G = np.dot(X.T, X) / M
    XtY = np.dot(X.T, Y) / M

    for target in prange(N):
        xty_target = XtY[:, target]
        current_beta = np.zeros(N, dtype=np.float64)

        for a_idx in range(n_alphas):
            alpha = alphas[a_idx]
            current_beta = numba_single_lasso_with_warm_start(
                G, xty_target, alpha, current_beta, max_iter, tol
            )
            coefs[target, :, a_idx] = current_beta

    return coefs


class NumbaLassoVAR1Evaluator:
    """
    An OOP evaluator that runs walk-forward out-of-sample predictions
    for VAR(1) LASSO using parallelized Numba coordinate descent.
    """
    def __init__(self, df, window_len=500, val_len=600, test_len=200, alphas=None, tol_rel=1e-4):
        self.df = df
        self.window_len = window_len
        self.val_len = val_len
        self.test_len = test_len

        self.rv5_arr = np.ascontiguousarray(df.to_numpy(dtype=np.float64))
        self.X = self.rv5_arr[:-1]
        self.Y = self.rv5_arr[1:]
        self.T_total, self.N = self.X.shape

        self.n_oos = val_len + test_len
        self.t_start = self.T_total - self.n_oos

        if self.t_start < window_len:
            raise ValueError(f"Insufficient observations for window_len ({window_len}) + val_len ({val_len}) + test_len ({test_len}).")

        if alphas is not None:
            self.alphas = np.array(alphas, dtype=np.float64)
        else:
            self.alphas = np.array([0.15, 0.08, 0.04, 0.025, 0.015], dtype=np.float64)

        self.n_alphas = len(self.alphas)
        self.tol = tol_rel
        self.eval_dates = df.index[self.t_start + 1 : self.t_start + 1 + self.n_oos]

    def run_eval(self, mode="rolling"):
        print(f"Starting {mode.upper()} window walk-forward VAR(1) LASSO (Evaluation on UNSTANDARDIZED RAW scale)...")

        t0 = time.perf_counter()
        preds_all_raw = np.zeros((self.n_oos, self.n_alphas, self.N), dtype=np.float64)
        actuals_all_raw = np.zeros((self.n_oos, self.N), dtype=np.float64)

        for step in range(self.n_oos):
            t = self.t_start + step
            train_start = 0 if mode == "expanding" else (t - self.window_len)

            X_window = self.X[train_start:t]
            Y_train = self.Y[train_start:t]

            X_mean = X_window.mean(axis=0)
            X_std = X_window.std(axis=0)
            X_std[X_std == 0.0] = 1.0
            X_train_scaled = (X_window - X_mean) / X_std
            X_today_scaled = (self.X[t] - X_mean) / X_std

            Y_mean = Y_train.mean(axis=0)
            Y_std = Y_train.std(axis=0)
            Y_std[Y_std == 0.0] = 1.0
            Y_train_scaled = (Y_train - Y_mean) / Y_std

            # Save the unscaled actuals
            actuals_all_raw[step] = self.Y[t]

            coefs = numba_parallel_lasso(X_train_scaled, Y_train_scaled, self.alphas, max_iter=1000, tol=self.tol)

            # Predict on standard scale, then unstandardize
            preds_std = coefs.transpose(2, 0, 1) @ X_today_scaled
            preds_all_raw[step] = preds_std * Y_std[None, :] + Y_mean[None, :]

        elapsed = time.perf_counter() - t0
        print(f"{mode.upper()} window run completed in: {elapsed:.2f}s")

        val_rmse = np.sqrt(np.mean((preds_all_raw[:self.val_len] - actuals_all_raw[:self.val_len, None, :]) ** 2, axis=(0, 2)))

        best_idx = int(np.argmin(val_rmse))
        best_alpha = self.alphas[best_idx]
        test_preds = preds_all_raw[self.val_len:, best_idx, :]
        test_actuals = actuals_all_raw[self.val_len:]
        test_rmse = np.sqrt(np.mean((test_preds - test_actuals) ** 2))
        test_mae = np.mean(np.abs(test_preds - test_actuals))

        return {
            "preds_all": preds_all_raw,
            "val_rmse": val_rmse,
            "best_alpha": best_alpha,
            "test_preds": test_preds,
            "test_rmse": test_rmse,
            "test_mae": test_mae
        }


class TensorAR1VAR1Evaluator:
    """
    A PyTorch-accelerated OOP evaluator to compute AR(1) and VAR(1) models
    with both rolling and expanding windows on standardized z-score variables,
    but evaluates performance on the original raw scale.
    """
    def __init__(self, df, window_len=500, val_len=100, test_len=20):
        self.df = df
        self.window_len = window_len
        self.val_len = val_len
        self.test_len = test_len

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.raw_data = torch.tensor(df.values, dtype=torch.float64, device=self.device)
        self.T_total, self.N = self.raw_data.shape

        self.n_oos = val_len + test_len
        self.t_start = self.T_total - self.n_oos

        if self.t_start < window_len:
            raise ValueError(f"Insufficient history for window_len ({window_len}) + val_len ({val_len}) + test_len ({test_len}).")

        self.eval_dates = df.index[self.t_start : self.t_start + self.n_oos]

    def run_eval(self, model_type="AR1", window_mode="rolling"):
        print(f"Running {window_mode.upper()} window {model_type} evaluation on {self.device} (Raw Scale Evaluation)...")
        t0 = time.perf_counter()

        preds_raw = np.zeros((self.n_oos, self.N), dtype=np.float64)
        actuals_raw = np.zeros((self.n_oos, self.N), dtype=np.float64)

        for step in range(self.n_oos):
            t = self.t_start + step
            train_start = 0 if window_mode == "expanding" else (t - self.window_len)

            X_win = self.raw_data[train_start:t]

            mean_t = X_win.mean(dim=0, keepdim=True)
            std_t = X_win.std(dim=0, keepdim=True)
            std_t[std_t == 0.0] = 1.0

            X_norm = (X_win - mean_t) / std_t

            # Retain the raw actuals
            actuals_raw[step] = self.raw_data[t : t+1].cpu().numpy()[0]

            F_train = X_norm[:-1, :]
            Y_train = X_norm[1:, :]
            f_latest = X_norm[-1:, :]

            if model_type == "AR1":
                num = (F_train * Y_train).sum(dim=0)
                den = (F_train ** 2).sum(dim=0)
                den[den == 0.0] = 1.0
                Beta = num / den
                pred_std = f_latest * Beta
            elif model_type == "VAR1":
                FtF = F_train.T @ F_train
                FtY = F_train.T @ Y_train
                FtF_stabilized = FtF + 1e-3 * torch.eye(self.N, device=self.device, dtype=torch.float64)
                Beta = torch.linalg.solve(FtF_stabilized, FtY)
                pred_std = f_latest @ Beta
            else:
                raise ValueError("Supported model_type parameters: 'AR1' or 'VAR1'")

            # Unstandardize prediction
            pred_unscaled = pred_std * std_t + mean_t
            preds_raw[step] = pred_unscaled.cpu().numpy()[0]

        elapsed = time.perf_counter() - t0
        print(f"{window_mode.upper()} {model_type} evaluation completed in: {elapsed:.2f}s")

        val_preds, val_actuals = preds_raw[:self.val_len], actuals_raw[:self.val_len]
        test_preds, test_actuals = preds_raw[self.val_len:], actuals_raw[self.val_len:]

        val_rmse = np.sqrt(np.mean((val_preds - val_actuals) ** 2))
        val_mae = np.mean(np.abs(val_preds - val_actuals))
        test_rmse = np.sqrt(np.mean((test_preds - test_actuals) ** 2))
        test_mae = np.mean(np.abs(test_preds - test_actuals))

        return {
            "preds_all": preds_raw,
            "actuals_all": actuals_raw,
            "val_rmse": val_rmse,
            "val_mae": val_mae,
            "test_rmse": test_rmse,
            "test_mae": test_mae
        }


In [52]:
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

def run_unified_comparison_pipeline(
    df,
    window_len=500,
    val_len=100,
    test_len=20,
    alphas=None,
    r_max=40
):
    """
    Runs all predictive models sequentially from scratch using a consistent
    validation and out-of-sample test split, reporting both the validation-phase
    hyperparameter metrics (detailed RMSE/MAE for each r and alpha) and the final test performance.
    """
    print("=== Starting Consolidated Out-of-Sample Evaluation Pipeline ===")
    t_start_pipeline = time.perf_counter()

    if alphas is None:
        alphas = [0.150, 0.100, 0.075, 0.055, 0.045, 0.040, 0.035, 0.028, 0.022, 0.015]
    alphas = np.array(alphas, dtype=np.float64)

    T_total, N = df.shape
    n_oos = val_len + test_len
    t_start = T_total - n_oos

    if t_start < window_len:
        raise ValueError(f"Not enough history! window_len={window_len} + val_len={val_len} + test_len={test_len} is larger than total days ({T_total})")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    X_raw = torch.tensor(df.values, dtype=torch.float32, device=device)

    # 1. Pure OLS Baseline (AR1) & Multivariate VAR(1) Baseline
    print(f"\n[1/4] Running Pure OLS & Multivariate VAR(1) Baselines ({n_oos} steps on {device})...")
    ar1_var1_eval = TensorAR1VAR1Evaluator(df, window_len=window_len, val_len=val_len, test_len=test_len)

    res_ar1_roll = ar1_var1_eval.run_eval(model_type="AR1", window_mode="rolling")
    res_var1_roll = ar1_var1_eval.run_eval(model_type="VAR1", window_mode="rolling")
    res_ar1_exp = ar1_var1_eval.run_eval(model_type="AR1", window_mode="expanding")
    res_var1_exp = ar1_var1_eval.run_eval(model_type="VAR1", window_mode="expanding")

    ar1_rmse_roll, ar1_mae_roll = res_ar1_roll["test_rmse"], res_ar1_roll["test_mae"]
    var1_rmse_roll, var1_mae_roll = res_var1_roll["test_rmse"], res_var1_roll["test_mae"]
    ar1_rmse_exp, ar1_mae_exp = res_ar1_exp["test_rmse"], res_ar1_exp["test_mae"]
    var1_rmse_exp, var1_mae_exp = res_var1_exp["test_rmse"], res_var1_exp["test_mae"]

    # 2. SVD Factor Models
    print(f"\n[2/4] Running Factor Models (Max r={r_max}) from scratch...")
    factor_evaluator = FactorModelEvaluator(df, r_max=r_max, n_eval=n_oos, window_size=window_len)
    exp_errors = factor_evaluator.compute_expanding_errors()
    roll_errors = factor_evaluator.compute_rolling_errors()

    # Extract entire Validation Performance Breakdown (r-wise)
    exp_val_metrics = factor_evaluator.evaluate_performance(exp_errors, start_idx=0, end_idx=val_len)
    roll_val_metrics = factor_evaluator.evaluate_performance(roll_errors, start_idx=0, end_idx=val_len)

    best_exp_r = int(exp_val_metrics['RMSE'].idxmin())
    best_roll_r = int(roll_val_metrics['RMSE'].idxmin())

    # Factor Validation Breakdown
    factor_val_breakdown = pd.DataFrame({
        'Expanding_Val_RMSE': exp_val_metrics['RMSE'],
        'Expanding_Val_MAE': exp_val_metrics['MAE'],
        'Rolling_Val_RMSE': roll_val_metrics['RMSE'],
        'Rolling_Val_MAE': roll_val_metrics['MAE']
    })

    # Complete test metrics for selected optimal models
    exp_test_all_metrics = factor_evaluator.evaluate_performance(exp_errors, start_idx=val_len, end_idx=n_oos)
    roll_test_all_metrics = factor_evaluator.evaluate_performance(roll_errors, start_idx=val_len, end_idx=n_oos)

    test_rmse_exp = exp_test_all_metrics.loc[best_exp_r, 'RMSE']
    test_mae_exp = exp_test_all_metrics.loc[best_exp_r, 'MAE']
    test_rmse_roll = roll_test_all_metrics.loc[best_roll_r, 'RMSE']
    test_mae_roll = roll_test_all_metrics.loc[best_roll_r, 'MAE']

    # 3. Parallelized Numba LASSO Models
    print("\n[3/4] Running Numba Parallelized VAR(1) LASSO from scratch...")
    lasso_evaluator = NumbaLassoVAR1Evaluator(df, window_len=window_len, val_len=val_len, test_len=test_len, alphas=alphas)
    res_rolling = lasso_evaluator.run_eval(mode="rolling")
    res_expanding = lasso_evaluator.run_eval(mode="expanding")

    # Extract actual raw outcomes during validation period (since predictions are now unstandardized)
    actuals_val_raw = df.iloc[t_start : t_start + val_len].values

    # Build detailed Validation Breakdown for Lasso
    expanding_val_rmses = []
    expanding_val_maes = []
    rolling_val_rmses = []
    rolling_val_maes = []
    for a_idx in range(len(alphas)):
        exp_preds_val = res_expanding['preds_all'][:val_len, a_idx, :]
        roll_preds_val = res_rolling['preds_all'][:val_len, a_idx, :]
        expanding_val_rmses.append(np.sqrt(np.mean((exp_preds_val - actuals_val_raw) ** 2)))
        expanding_val_maes.append(np.mean(np.abs(exp_preds_val - actuals_val_raw)))
        rolling_val_rmses.append(np.sqrt(np.mean((roll_preds_val - actuals_val_raw) ** 2)))
        rolling_val_maes.append(np.mean(np.abs(roll_preds_val - actuals_val_raw)))

    lasso_val_breakdown = pd.DataFrame({
        'Alpha': alphas,
        'Expanding_Val_RMSE': expanding_val_rmses,
        'Expanding_Val_MAE': expanding_val_maes,
        'Rolling_Val_RMSE': rolling_val_rmses,
        'Rolling_Val_MAE': rolling_val_maes
    }).set_index('Alpha')

    lasso_rmse_exp = res_expanding['test_rmse']
    lasso_mae_exp = res_expanding['test_mae']
    lasso_rmse_roll = res_rolling['test_rmse']
    lasso_mae_roll = res_rolling['test_mae']

    # 4. Consolidate Results
    print("\n[4/4] Consolidating metrics and constructing comparisons...")
    results_dict = {
        'AR(1) (Rolling)': [ar1_rmse_roll, ar1_mae_roll],
        'AR(1) (Expanding)': [ar1_rmse_exp, ar1_mae_exp],
        'VAR(1) Baseline (Rolling)': [var1_rmse_roll, var1_mae_roll],
        'VAR(1) Baseline (Expanding)': [var1_rmse_exp, var1_mae_exp],
        f'Factor Model (Rolling, r={best_roll_r})': [test_rmse_roll, test_mae_roll],
        f'Factor Model (Expanding, r={best_exp_r})': [test_rmse_exp, test_mae_exp],
        'VAR(1) LASSO (Rolling)': [lasso_rmse_roll, lasso_mae_roll],
        'VAR(1) LASSO (Expanding)': [lasso_rmse_exp, lasso_mae_exp]
    }

    comparison_summary = pd.DataFrame.from_dict(results_dict, orient='index', columns=['Out-of-Sample RMSE', 'Out-of-Sample MAE']).sort_values('Out-of-Sample RMSE')

    print("\n=== SELECTED HYPERPARAMETERS (Validation Phase) ===")
    print(f"[-] Expanding Factor Model: Selected r = {best_exp_r} (Val RMSE = {exp_val_metrics.loc[best_exp_r, 'RMSE']:.5f}, Val MAE = {exp_val_metrics.loc[best_exp_r, 'MAE']:.5f})")
    print(f"[-] Rolling Factor Model:   Selected r = {best_roll_r} (Val RMSE = {roll_val_metrics.loc[best_roll_r, 'RMSE']:.5f}, Val MAE = {roll_val_metrics.loc[best_roll_r, 'MAE']:.5f})")
    print(f"[-] Expanding VAR(1) LASSO: Selected alpha = {res_expanding['best_alpha']:.5f} (Val RMSE = {res_expanding['val_rmse'].min():.5f})")
    print(f"[-] Rolling VAR(1) LASSO:   Selected alpha = {res_rolling['best_alpha']:.5f} (Val RMSE = {res_rolling['val_rmse'].min():.5f})")

    print("\n=== [TRAINING/VALIDATION PERIOD] Metrics per Factor Count (r) ===")
    display(factor_val_breakdown)

    print("\n=== [TRAINING/VALIDATION PERIOD] Metrics per LASSO Penalty (alpha) ===")
    display(lasso_val_breakdown)

    print("\n=== [FINAL TEST PERIOD] Out-of-Sample Model Comparison ===")
    display(comparison_summary)

    # --- Plotting the Final Comparison Graphs ---
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    comparison_summary_sorted_rmse = comparison_summary.sort_values('Out-of-Sample RMSE')
    comparison_summary_sorted_mae = comparison_summary.sort_values('Out-of-Sample MAE')

    axes[0].barh(comparison_summary_sorted_rmse.index, comparison_summary_sorted_rmse['Out-of-Sample RMSE'], color='skyblue')
    axes[0].set_title('Out-of-Sample Test RMSE Comparison')
    axes[0].set_xlabel('RMSE')

    axes[1].barh(comparison_summary_sorted_mae.index, comparison_summary_sorted_mae['Out-of-Sample MAE'], color='lightgreen')
    axes[1].set_title('Out-of-Sample Test MAE Comparison')
    axes[1].set_xlabel('MAE')

    plt.tight_layout()
    plt.show()

    print(f"Total pipeline process took: {time.perf_counter() - t_start_pipeline:.2f}s")
    return comparison_summary, factor_val_breakdown, lasso_val_breakdown


In [ ]:
# Run with updated evaluation logic displaying detailed training/validation breakdowns along with the test metrics
results = run_unified_comparison_pipeline(
    X_df,
    window_len=500,
    val_len=300,
    test_len=100,
    alphas=[0.150, 0.100, 0.075, 0.055, 0.045, 0.040, 0.035, 0.028, 0.022, 0.015],
    r_max=100
)

=== Starting Consolidated Out-of-Sample Evaluation Pipeline ===

[1/4] Running Pure OLS & Multivariate VAR(1) Baselines (400 steps on cpu)...
Running ROLLING window AR1 evaluation on cpu (Raw Scale Evaluation)...
ROLLING AR1 evaluation completed in: 0.32s
Running ROLLING window VAR1 evaluation on cpu (Raw Scale Evaluation)...
ROLLING VAR1 evaluation completed in: 4.70s
Running EXPANDING window AR1 evaluation on cpu (Raw Scale Evaluation)...
EXPANDING AR1 evaluation completed in: 1.44s
Running EXPANDING window VAR1 evaluation on cpu (Raw Scale Evaluation)...
EXPANDING VAR1 evaluation completed in: 11.44s

[2/4] Running Factor Models (Max r=100) from scratch...
Computing expanding window errors on: cpu
Computing rolling window errors on: cpu


In [36]:
import pandas as pd
# Print the exact lasso validation breakdown DataFrame stored in the results tuple
display(results[2])

,Expanding_Val_RMSE,Expanding_Val_MAE,Rolling_Val_RMSE,Rolling_Val_MAE
Alpha,,,,
0.150,0.421279,0.218855,0.421159,0.218819
0.100,0.424851,0.220993,0.424628,0.220822
0.075,0.429458,0.225440,0.429127,0.225091
0.055,0.436805,0.231893,0.436275,0.231334
0.045,0.443490,0.239551,0.442738,0.238749
0.040,0.449703,0.246784,0.448791,0.245830
0.035,0.458189,0.256560,0.457111,0.255359
0.028,0.472534,0.273983,0.471045,0.272216
0.022,0.486144,0.290999,0.484262,0.288635


In [ ]:
import time
import numpy as np
from sklearn.linear_model import Lasso

# --- Diagnostic & Robustness Check of Numba coordinate descent ---
def run_diagnostics():
    print("=== Running Numba Coordinate Descent Diagnostics ===")

    # Generate a small synthetic dataset representing one step of the loop
    np.random.seed(42)
    M, N = 500, 40
    X_test = np.ascontiguousarray(np.random.randn(M, N))
    y_test = np.ascontiguousarray(np.random.randn(M))

    # Precompute Gram matrix and cross correlation
    G = np.dot(X_test.T, X_test) / M
    xty = np.dot(X_test.T, y_test) / M

    # Choose an intermediate alpha value
    alpha = 0.05
    beta_init = np.zeros(N, dtype=np.float64)

    # Time Numba single solver (first run has compile overhead, second is raw speed)
    _ = numba_single_lasso_with_warm_start(G, xty, alpha, beta_init, max_iter=1000, tol=1e-5)

    t0 = time.perf_counter()
    beta_numba = numba_single_lasso_with_warm_start(G, xty, alpha, beta_init, max_iter=1000, tol=1e-5)
    t_numba = time.perf_counter() - t0

    # Compare with Scikit-Learn's highly optimized C-based Lasso solver
    clf = Lasso(alpha=alpha, fit_intercept=False, max_iter=1000, tol=1e-5)
    t1 = time.perf_counter()
    clf.fit(X_test, y_test)
    t_sklearn = time.perf_counter() - t1
    beta_sklearn = clf.coef_

    # Compute metrics
    max_diff = np.max(np.abs(beta_numba - beta_sklearn))
    sparsity_numba = np.sum(beta_numba == 0.0)

    print(f"Numba Single Solver Execution Time: {t_numba:.6f} seconds")
    print(f"Sklearn Solver Execution Time:      {t_sklearn:.6f} seconds")
    print(f"Maximum coefficient discrepancy:    {max_diff:.2e}")
    print(f"Identified Sparsity (Zero Coefs):   {sparsity_numba} out of {N}")

    if max_diff > 1e-3:
        print("WARNING: Divergence or mathematical error detected between Numba and Sklearn!")
    else:
        print("SUCCESS: Mathematical convergence and solution matching verified.")

run_diagnostics()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. Build a summary table using the validation phase metrics
summary = pd.DataFrame({
    'Method': ['Rolling (1000d)', 'Expanding'],
    'Best r (Val RMSE)': [
        roll_val_metrics['RMSE'].idxmin(),
        exp_val_metrics['RMSE'].idxmin(),
    ],
    'Min Val RMSE': [
        roll_val_metrics['RMSE'].min(),
        exp_val_metrics['RMSE'].min(),
    ],
    'Best r (Val MAE)': [
        roll_val_metrics['MAE'].idxmin(),
        exp_val_metrics['MAE'].idxmin(),
    ],
    'Min Val MAE': [
        roll_val_metrics['MAE'].min(),
        exp_val_metrics['MAE'].min(),
    ],
}).set_index('Method')

print('=== Validation Split Performance Comparison (First 600 Days) ===')
display(summary)

# 2. Side-by-side validation error curves (RMSE & MAE) to show how optimal r was selected
fig, axes = plt.subplots(1, 2, figsize=(15, 6), sharex=True)

# --- Subplot 1: Validation RMSE ---
r_roll_rmse = roll_val_metrics['RMSE'].idxmin()
v_roll_rmse = roll_val_metrics.loc[r_roll_rmse, 'RMSE']
r_exp_rmse = exp_val_metrics['RMSE'].idxmin()
v_exp_rmse = exp_val_metrics.loc[r_exp_rmse, 'RMSE']

axes[0].plot(
    roll_val_metrics.index,
    roll_val_metrics['RMSE'],
    marker='o',
    markersize=4,
    color='tab:red',
    label='Rolling Val RMSE',
)
axes[0].axvline(
    r_roll_rmse,
    color='tab:red',
    linestyle=':',
    linewidth=1.5,
    label=f'Rolling Min: r={r_roll_rmse} ({v_roll_rmse:.4f})',
)
axes[0].scatter([r_roll_rmse], [v_roll_rmse], color='tab:red', s=70, zorder=5)

axes[0].plot(
    exp_val_metrics.index,
    exp_val_metrics['RMSE'],
    marker='s',
    markersize=4,
    color='tab:blue',
    label='Expanding Val RMSE',
)
axes[0].axvline(
    r_exp_rmse,
    color='tab:blue',
    linestyle='--',
    linewidth=1.5,
    label=f'Expanding Min: r={r_exp_rmse} ({v_exp_rmse:.4f})',
)
axes[0].scatter([r_exp_rmse], [v_exp_rmse], color='tab:blue', s=70, zorder=5)

axes[0].set_title('Validation Window RMSE (First 600 Days)')
axes[0].set_xlabel('Number of Factors (r)')
axes[0].set_ylabel('RMSE')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# --- Subplot 2: Validation MAE ---
r_roll_mae = roll_val_metrics['MAE'].idxmin()
v_roll_mae = roll_val_metrics.loc[r_roll_mae, 'MAE']
r_exp_mae = exp_val_metrics['MAE'].idxmin()
v_exp_mae = exp_val_metrics.loc[r_exp_mae, 'MAE']

axes[1].plot(
    roll_val_metrics.index,
    roll_val_metrics['MAE'],
    marker='o',
    markersize=4,
    color='tab:red',
    label='Rolling Val MAE',
)
axes[1].axvline(
    r_roll_mae,
    color='tab:red',
    linestyle=':',
    linewidth=1.5,
    label=f'Rolling Min: r={r_roll_mae} ({v_roll_mae:.4f})',
)
axes[1].scatter([r_roll_mae], [v_roll_mae], color='tab:red', s=70, zorder=5)

axes[1].plot(
    exp_val_metrics.index,
    exp_val_metrics['MAE'],
    marker='s',
    markersize=4,
    color='tab:blue',
    label='Expanding Val MAE',
)
axes[1].axvline(
    r_exp_mae,
    color='tab:blue',
    linestyle='--',
    linewidth=1.5,
    label=f'Expanding Min: r={r_exp_mae} ({v_exp_mae:.4f})',
)
axes[1].scatter([r_exp_mae], [v_exp_mae], color='tab:blue', s=70, zorder=5)

axes[1].set_title('Validation Window MAE (First 600 Days)')
axes[1].set_xlabel('Number of Factors (r)')
axes[1].set_ylabel('MAE')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()